# Fit a model

**The job.** Same data, two questions. Predict a number, and predict a label.

The point here is that both are the *same graph*. Only the last two steps
change. Swapping regression for classification is a change of route, not a
rewrite.

Everything is numpy. No scikit-learn. The maths is short enough to read, and
you can see there is nothing hidden in it.

**In:** a generated dataset.
**Out:** scores for both jobs, and predictions.
**Files:** metrics.json, predictions.csv.

In [1]:
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import json, pathlib
from dataclasses import replace

from browsergraph import execute, viz
from browsergraph.compile import compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import Edge, NodeCandidate, StageDefinition, WorkbenchDefinition

# A fresh folder each run. Left-over files from a previous run make the "what
# did this produce" list a lie, and that list is half the point here.
import shutil
WORK = pathlib.Path("work")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

def node(node_id, capability, takes, gives, **kw):
    """Describe one node. Ports are (name, type) pairs."""
    return NodeManifest(
        id=node_id, kind="function", description=f"{capability} via {node_id}",
        capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in takes),
        outputs=tuple(PortSpec(n, t) for n, t in gives), **kw)

def stage(sid, name, takes, gives, capability, candidates):
    """Describe one step of the job, and what could do it."""
    return StageDefinition(
        id=sid, name=name, required_capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in takes),
        outputs=tuple(PortSpec(n, t) for n, t in gives),
        success=f"{name} produced its declared output",
        candidates=tuple(candidates))

def build(title, task, stages, nodes, edges=()):
    """Put it together and check it before anything runs."""
    bench = WorkbenchDefinition(
        title=title, task=task, stages=tuple(stages), nodes=tuple(nodes),
        edges=tuple(edges),
        candidates=tuple(NodeCandidate(id=n.id, node_id=n.id) for n in nodes))
    problems = bench.validate()
    print("problems:", problems if problems else "none")
    return bench

print("ready")

ready


## The input

800 rows. Three numeric columns and one category. The number we predict depends
on all of them plus noise. The label is just "is the number above average" —
so a model that predicts the number well should classify well too.

In [2]:
import numpy as np

rng = np.random.default_rng(11)
N = 800

size = rng.normal(70, 18, N).round(1)
age = rng.integers(1, 40, N)
rooms = rng.integers(1, 6, N)
area = rng.choice(["north", "south", "central"], N, p=[0.4, 0.35, 0.25])

premium = np.select([area == "central", area == "south"], [95.0, 40.0], 0.0)
price = (28 * size - 1.7 * age + 12 * rooms + premium
         + rng.normal(0, 45, N)).round(1)
above = (price > np.median(price)).astype(int)

print(f"{N} rows")
print(f"{'size':>8}{'age':>6}{'rooms':>7}{'area':>9}{'price':>10}{'above':>7}")
for i in range(5):
    print(f"{size[i]:>8.1f}{age[i]:>6}{rooms[i]:>7}{area[i]:>9}{price[i]:>10.1f}{above[i]:>7}")
print(f"\nprice: min {price.min():.0f}  median {np.median(price):.0f}  max {price.max():.0f}")

800 rows
    size   age  rooms     area     price  above
    70.6    27      5  central    2095.3      1
    94.5     4      2    south    2713.9      1
    92.0    26      3    north    2639.8      1
    60.8    24      4  central    1794.5      0
    64.6     9      4    south    1896.2      0

price: min 560  median 2023  max 3778


## The steps

Load, split, then two independent feature passes — numbers and the category —
that meet at the assemble step. Then fit and score.

Numbers and categories are genuinely independent. Neither waits on the other.
Drawn as a list that fact disappears.

In [3]:
nodes = [
    node("load.arrays",  "data.read",    [],                  [("out", "Frame")]),
    node("split.random", "data.split",   [("in", "Frame")],   [("train", "Frame"), ("valid", "Frame")]),
    node("num.standard", "feature.numeric",     [("in", "Frame")], [("out", "Matrix")]),
    node("num.raw",      "feature.numeric",     [("in", "Frame")], [("out", "Matrix")]),
    node("cat.onehot",   "feature.categorical", [("in", "Frame")], [("out", "Matrix")]),
    node("assemble.hstack","feature.assemble",  [("numeric", "Matrix"), ("categorical", "Matrix")], [("out", "Matrix")]),
    node("fit.leastsquares","model.fit", [("in", "Matrix")],  [("out", "Model")]),
    node("fit.logistic",    "model.fit", [("in", "Matrix")],  [("out", "Model")]),
    node("fit.ridge",       "model.fit", [("in", "Matrix")],  [("out", "Model")]),
    node("score.regression","model.score",[("in", "Model")],  [("out", "Score")]),
    node("score.classifier","model.score",[("in", "Model")],  [("out", "Score")]),
]

stages = [
    stage("load",     "Load the data",     [],                [("out", "Frame")], "data.read", ["load.arrays"]),
    stage("split",    "Hold some back",    [("in", "Frame")], [("train", "Frame"), ("valid", "Frame")], "data.split", ["split.random"]),
    stage("numeric",  "Scale the numbers", [("in", "Frame")], [("out", "Matrix")], "feature.numeric",     ["num.standard", "num.raw"]),
    stage("category", "Encode the area",   [("in", "Frame")], [("out", "Matrix")], "feature.categorical", ["cat.onehot"]),
    stage("assemble", "Put them together", [("numeric", "Matrix"), ("categorical", "Matrix")], [("out", "Matrix")], "feature.assemble", ["assemble.hstack"]),
    stage("fit",      "Fit a model",       [("in", "Matrix")], [("out", "Model")], "model.fit",   ["fit.leastsquares", "fit.logistic", "fit.ridge"]),
    stage("score",    "Score it",          [("in", "Model")],  [("out", "Score")], "model.score", ["score.regression", "score.classifier"]),
]

edges = [Edge("load", "split"), Edge("split", "numeric", from_port="train"),
         Edge("split", "category", from_port="train"),
         Edge("numeric", "assemble", to_port="numeric"),
         Edge("category", "assemble", to_port="categorical"),
         Edge("assemble", "fit"), Edge("fit", "score")]

bench = build("Fit a model", "Predict a number, and predict a label.",
              stages, nodes, edges)
print("layers:", bench.layers())
print("routes:", bench.route_count(), "— two ways to fit, two ways to score")

problems: none
layers: [['load'], ['split'], ['numeric', 'category'], ['assemble'], ['fit'], ['score']]
routes: 12 — two ways to fit, two ways to score


In [4]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1170 308" width="1170" height="308" style="max-width:none" role="img"><defs><marker id="bg53322804-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Load the data</text><text x="69" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="363.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="270" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="279" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Hold some back</text><text x="279" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="573.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2 · 2 parallel</text><g><rect x="480" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Scale the numbers</text><text x="489" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><g><rect x="480" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Encode the area</text><text x="489" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="783.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 3</text><g><rect x="690" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="699" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Put them together</text><text x="699" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="993.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 4</text><g><rect x="900" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="909" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Fit a model</text><text x="909" y="149.0" font-size="9.5" fill="#68737f">3 candidates</text></g><text x="1203.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 5</text><g><rect x="1110" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1119" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Score it</text><text x="1119" y="149.0" font-size="9.5" fill="#68737f">2 candidates</text></g><path d="M246,141.0 C258.0,141.0 258.0,141.0 270,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg53322804-arrow)"/><path d="M456,141.0 C468.0,141.0 468.0,100.0 480,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg53322804-arrow)"/><text x="468.0" y="115.5" text-anchor="middle" font-size="9" fill="#68737f">train</text><path d="M456,141.0 C468.0,141.0 468.0,182.0 480,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg53322804-arrow)"/><text x="468.0" y="156.5" text-anchor="middle" font-size="9" fill="#68737f">train</text><path d="M666,100.0 C678.0,100.0 678.0,141.0 690,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg53322804-arrow)"/><text x="678.0" y="115.5" text-anchor="middle" font-size="9" fill="#68737f">numeric</text><path d="M666,182.0 C678.0,182.0 678.0,141.0 690,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg53322804-arrow)"/><text x="678.0" y="156.5" text-anchor="middle" font-size="9" fill="#68737f">categorical</text><path d="M876,141.0 C888.0,141.0 888.0,141.0 900,141.0" fill="none" strok

## The code

`np.linalg.lstsq` for the regression. Plain gradient descent for the logistic
one. Both are a few lines, and both are doing the real thing.

In [5]:
def load_arrays():
    return {"size": size, "age": age, "rooms": rooms, "area": area,
            "price": price, "above": above}

def split_random(**kw):
    """Its own seeded generator, not the shared one.

    This started out drawing from the module-level `rng`, which advances every
    time anything uses it. Two effects, both bad. Re-running the same plan gave
    a different score, so the plan's own claim to be deterministic was false.
    And worse, comparing four routes below would have scored each one on a
    *different* split — which is not a comparison of models at all.
    """
    frame = kw["in"]
    order = np.random.default_rng(2024).permutation(N)
    cut = int(N * 0.75)
    take = lambda idx: {k: v[idx] for k, v in frame.items()}
    return {"train": take(order[:cut]), "valid": take(order[cut:])}

def num_standard(**kw):
    """Centre and scale, using the training numbers only."""
    frame = kw["in"]
    cols = np.column_stack([frame["size"], frame["age"], frame["rooms"]]).astype(float)
    mean, sd = cols.mean(0), cols.std(0)
    return {"matrix": (cols - mean) / sd, "mean": mean, "sd": sd,
            "target": frame["price"], "label": frame["above"]}

def cat_onehot(**kw):
    """One column per area. Three areas, three columns."""
    frame = kw["in"]
    areas = ["north", "south", "central"]
    matrix = np.column_stack([(frame["area"] == a).astype(float) for a in areas])
    return {"matrix": matrix, "areas": areas}

def assemble_hstack(**kw):
    numeric, categorical = kw["numeric"], kw["categorical"]
    X = np.column_stack([np.ones(len(numeric["matrix"])),
                         numeric["matrix"], categorical["matrix"]])
    return {"X": X, "y": numeric["target"], "label": numeric["label"],
            "names": ["bias", "size", "age", "rooms", *categorical["areas"]]}

def fit_leastsquares(**kw):
    data = kw["in"]
    weights, *_ = np.linalg.lstsq(data["X"], data["y"], rcond=None)
    return {"kind": "regression", "weights": weights, "data": data}

def num_raw(**kw):
    """No scaling at all. Kept as a real option so the search has something to
    reject on evidence rather than on somebody's opinion."""
    frame = kw["in"]
    cols = np.column_stack([frame["size"], frame["age"], frame["rooms"]]).astype(float)
    return {"matrix": cols, "mean": np.zeros(3), "sd": np.ones(3),
            "target": frame["price"], "label": frame["above"]}

def fit_ridge(**kw):
    """Least squares with a small penalty on big weights."""
    data = kw["in"]
    X, y = data["X"], data["y"]
    penalty = 1.0 * np.eye(X.shape[1])
    penalty[0, 0] = 0.0                       # never penalise the bias
    weights = np.linalg.solve(X.T @ X + penalty, X.T @ y)
    return {"kind": "regression", "weights": weights, "data": data}

def fit_logistic(**kw):
    """Gradient descent. 400 steps is plenty for this."""
    data = kw["in"]
    X, y = data["X"], data["label"].astype(float)
    w = np.zeros(X.shape[1])
    for _ in range(400):
        p = 1 / (1 + np.exp(-X @ w))
        w -= 0.5 * (X.T @ (p - y)) / len(y)
    return {"kind": "classification", "weights": w, "data": data}

def score_regression(**kw):
    m = kw["in"]
    X, y = m["data"]["X"], m["data"]["y"]
    pred = X @ m["weights"]
    ss_res = float(((y - pred) ** 2).sum())
    ss_tot = float(((y - y.mean()) ** 2).sum())
    return {"kind": "regression", "r2": 1 - ss_res / ss_tot,
            "mae": float(np.abs(y - pred).mean()),
            "prediction": pred, "truth": y,
            "weights": dict(zip(m["data"]["names"], m["weights"].round(2)))}

def score_classifier(**kw):
    m = kw["in"]
    X, y = m["data"]["X"], m["data"]["label"]
    prob = 1 / (1 + np.exp(-X @ m["weights"]))
    pred = (prob > 0.5).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    return {"kind": "classification",
            "accuracy": float((pred == y).mean()),
            "precision": tp / (tp + fp) if tp + fp else 0.0,
            "recall": tp / (tp + fn) if tp + fn else 0.0,
            "confusion": {"tp": tp, "fp": fp, "tn": tn, "fn": fn},
            "prediction": pred, "truth": y}

runtime = execute.Runtime({
    "load.arrays": load_arrays, "split.random": split_random,
    "num.standard": num_standard, "num.raw": num_raw, "cat.onehot": cat_onehot,
    "fit.ridge": fit_ridge,
    "assemble.hstack": assemble_hstack,
    "fit.leastsquares": fit_leastsquares, "fit.logistic": fit_logistic,
    "score.regression": score_regression, "score.classifier": score_classifier,
})
print("all steps have code:", runtime.missing(
    compile_route(bench, {**{s.id: s.candidates[0] for s in bench.leaf_stages}})) == [])

all steps have code: True


## Route one: predict the number

In [6]:
regression = {s.id: s.candidates[0] for s in bench.leaf_stages}
regression["fit"] = "fit.leastsquares"
regression["score"] = "score.regression"

plan_r = compile_route(bench, regression)
run_r = execute.run(plan_r, runtime)
print(run_r.text())

got = run_r.output("score")
print(f"\nR²  {got['r2']:.4f}      mean error {got['mae']:,.1f}")
print("\nwhat it learned:")
for name, weight in got["weights"].items():
    print(f"  {name:<9}{weight:>10.2f}")

plan plan:4b653d10844bfaed114d9…
7 steps in 0.002s — ok
  ok   load             0.000s  load.arrays
  ok   split            0.000s  split.random
  ok   numeric          0.000s  num.standard
  ok   category         0.000s  cat.onehot
  ok   assemble         0.000s  assemble.hstack
  ok   fit              0.000s  fit.leastsquares
  ok   score            0.000s  score.regression

R²  0.9917      mean error 35.4

what it learned:
  bias        1514.68
  size         494.15
  age          -22.01
  rooms         16.45
  north        458.73
  south        497.14
  central      558.81


The weights line up with how the data was made: `size` is the big driver,
`central` is worth more than `south`, and `age` pulls down. That is a check on
the pipeline, not just on the model.

## Route two: predict the label

Same graph. Two different candidates.

In [7]:
classification = dict(regression)
classification["fit"] = "fit.logistic"
classification["score"] = "score.classifier"

plan_c = compile_route(bench, classification)
run_c = execute.run(plan_c, runtime)

got_c = run_c.output("score")
print(f"accuracy {got_c['accuracy']:.3f}   precision {got_c['precision']:.3f}   recall {got_c['recall']:.3f}")
c = got_c["confusion"]
print(f"\n            predicted 0   predicted 1")
print(f"  actual 0 {c['tn']:>12} {c['fp']:>13}")
print(f"  actual 1 {c['fn']:>12} {c['tp']:>13}")
print(f"\nsame graph? {plan_r.layers == plan_c.layers}")
print(f"different plan? {plan_r.digest != plan_c.digest}")

accuracy 0.977   precision 0.974   recall 0.980

            predicted 0   predicted 1
  actual 0          290             8
  actual 1            6           296

same graph? True
different plan? True


Same layers, different digest. The shape of the work did not change. What ran
inside it did, and the digest proves the two results came from different graphs
so they can never be mixed up later.

## Let the evidence pick, instead of picking yourself

So far I chose the route by hand. Fine for two options. There are now six ways
to predict the number — two ways to handle the numbers, three ways to fit — and
picking by hand stops being a plan.

So run them all and let the measured result decide. Not a prior. Not an opinion
about which model is better. The actual score on this actual data.

In [8]:
import itertools

numeric_options = bench.stage("numeric").candidates
fit_options = [c for c in bench.stage("fit").candidates if c != "fit.logistic"]

results = []
for numeric, fit in itertools.product(numeric_options, fit_options):
    trial = dict(regression, numeric=numeric, fit=fit, score="score.regression")
    plan_t = compile_route(bench, trial)
    got_t = execute.run(plan_t, runtime)
    if not got_t.ok:
        results.append((numeric, fit, None, None, plan_t.digest, got_t.steps[-1].error))
        continue
    s_t = got_t.output("score")
    results.append((numeric, fit, s_t["r2"], s_t["mae"], plan_t.digest, ""))

print(f"{'numbers':<14}{'model':<18}{'R2':>9}{'mean error':>13}   plan")
for numeric, fit, r2, mae, digest, err in results:
    if r2 is None:
        print(f"{numeric:<14}{fit:<18}{'failed':>9}{'':>13}   {err[:34]}")
    else:
        print(f"{numeric:<14}{fit:<18}{r2:>9.4f}{mae:>13,.1f}   {digest[5:17]}")

numbers       model                    R2   mean error   plan
num.standard  fit.leastsquares     0.9917         35.4   4b653d10844b
num.standard  fit.ridge            0.9917         35.4   a152bac53fe9
num.raw       fit.leastsquares     0.9917         35.4   b366ee7200dc
num.raw       fit.ridge            0.9917         35.4   f1361142feb3


## The winner, and how much of the space that took

In [9]:
ranked = sorted([r for r in results if r[2] is not None], key=lambda r: -r[2])
best = ranked[0]
print(f"best:  {best[0]:<13}+ {best[1]:<18}R2 {best[2]:.4f}   plan {best[4][5:17]}")
print(f"worst: {ranked[-1][0]:<13}+ {ranked[-1][1]:<18}R2 {ranked[-1][2]:.4f}")
print(f"\ngap between best and worst: {best[2] - ranked[-1][2]:.4f} R2")
print("\nEvery number above was measured. None of it was a prior.")

best:  num.standard + fit.leastsquares  R2 0.9917   plan 4b653d10844b
worst: num.standard + fit.ridge         R2 0.9917

gap between best and worst: 0.0000 R2

Every number above was measured. None of it was a prior.


In [10]:
viz.funnel([
    ("every route in the graph", bench.route_count()),
    ("regression routes",        len(results)),
    ("ran without failing",      len(ranked)),
    ("chosen",                   1),
], title="how the model was picked")

Figure(svg='<svg viewBox="0 0 1000 296" width="1000" height="296" style="max-width:none" role="img"><text x="176" y="83" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">every route in the graph</text><rect x="190" y="66" width="670.0" height="26" rx="4" fill="#2d6cb5" opacity="0.72" stroke="#2d6cb5" stroke-width="1"/><text x="870.0" y="83" font-size="11" fill="#22303f">12</text><text x="176" y="129" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">regression routes</text><rect x="190" y="112" width="420.4" height="26" rx="4" fill="#2d6cb5" opacity="0.53" stroke="#2d6cb5" stroke-width="1"/><text x="620.4" y="129" font-size="11" fill="#22303f">4</text><text x="684.4" y="129" font-size="10" fill="#68737f">÷3</text><text x="176" y="175" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">ran without failing</text><rect x="190" y="158" width="420.4" height="26" rx="4" fill="#2d6cb5" opacity="0.53" stroke="#2d6cb5" stroke-width="1"/><text x="620.4" y="175" font-size="11" fill="#22303f">4</text><text x="684.4" y="175" font-size="10" fill="#68737f">÷1</text><text x="176" y="221" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">chosen</text><rect x="190" y="204" width="181.1" height="26" rx="4" fill="#1f8a4c" opacity="0.36" stroke="#1f8a4c" stroke-width="1"/><text x="381.1" y="221" font-size="11" fill="#22303f">1</text><text x="445.1" y="221" font-size="10" fill="#68737f">÷4</text><text x="190" y="278" font-size="9.5" fill="#68737f">bar length is log-scaled; labels are exact counts</text></svg>', title='how the model was picked', note='Every row is a real filter, in order.', width=1000, height=296)

### Read that result honestly

The gap between best and worst is **0.0000**. On this data, with this split,
the choice makes no measurable difference at all.

So the right conclusion is not "num.standard won". It is "this decision does not
matter here, so stop spending time on it". Scaling does nothing for least
squares because least squares is scale-invariant, and the ridge penalty is too
small to bite. Both of those are true facts about the maths, and the measurement
agrees with them.

A search that always announces a winner will always find one. The useful search
tells you when the winner is noise.

Four routes tried, four measured, one picked. Small enough to enumerate, and the
notebook says so rather than implying a bigger search happened.

When the space is too big to enumerate, `browsergraph.search` does this with a
beam and reports how much of the space it covered. It refuses to enumerate a
space it cannot finish, rather than trying and running out of memory — which is
exactly what it used to do.

In [11]:
best_route = dict(regression, numeric=best[0], fit=best[1])
plan_best = compile_route(bench, best_route)
run_best = execute.run(plan_best, runtime)
print(f"re-ran the winner: R2 {run_best.output('score')['r2']:.4f}"
      f"   same plan: {plan_best.digest == best[4]}")

re-ran the winner: R2 0.9917   same plan: True


## Save the results

In [12]:
import csv

def write_results(workspace, **kw):
    metrics = {"regression": {k: v for k, v in kw["reg"].items()
                              if k not in ("prediction", "truth", "weights")},
               "classification": {k: v for k, v in kw["clf"].items()
                                  if k not in ("prediction", "truth")},
               "regression_weights": {k: float(v) for k, v in kw["reg"]["weights"].items()},
               "plans": {"regression": kw["reg_plan"], "classification": kw["clf_plan"]}}
    (workspace / "metrics.json").write_text(json.dumps(metrics, indent=2, default=float))

    path = workspace / "predictions.csv"
    with path.open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["actual_price", "predicted_price", "actual_label", "predicted_label"])
        for row in zip(kw["reg"]["truth"], kw["reg"]["prediction"],
                       kw["clf"]["truth"], kw["clf"]["prediction"]):
            writer.writerow([f"{row[0]:.1f}", f"{row[1]:.1f}", int(row[2]), int(row[3])])
    return {"rows": len(kw["reg"]["truth"])}

wrote = write_results(WORK, reg=got, clf=got_c,
                      reg_plan=plan_r.digest, clf_plan=plan_c.digest)
print(wrote)
for name in ("metrics.json", "predictions.csv"):
    path = WORK / name
    print(f"  {name:<18}{path.stat().st_size:>8,} bytes")
print()
print((WORK / "predictions.csv").read_text().splitlines()[0])
for line in (WORK / "predictions.csv").read_text().splitlines()[1:6]:
    print(line)

{'rows': 600}


  metrics.json           675 bytes
  predictions.csv     11,444 bytes

actual_price,predicted_price,actual_label,predicted_label
2700.6,2613.5,1,1
2308.3,2256.2,1,1
1508.2,1530.3,0,0
1359.6,1438.2,0,0
2146.2,2139.5,1,1


## What the five notebooks showed

Same library, five jobs that have nothing in common:

1. a web page turned into rows,
2. mixed records forced into one schema,
3. images checked and resized,
4. a messy table cleaned with a record of every change,
5. a model fitted two different ways.

Each one had real input, ran real code, and left files behind. The graph was
written the same way every time, the checks were the same checks, and the
picture was drawn by the same function.